In [34]:
%%capture
pip install transformer_lens transformers jaxtyping tiktoken

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [2]:
#from src.utils import get_current_time_str
#from src.utils import get_repo_root
import numpy as np
from tqdm import tqdm

from transformer_lens import HookedTransformer
from transformer_lens.hook_points import HookPoint
from transformers import AutoTokenizer
from torch import Tensor
from jaxtyping import Float, Int
import torch

/opt/homebrew/Caskroom/miniconda/base/envs/algo-neutrality/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def getDevice():
    if torch.cuda.is_available(): #nvidia/runpod
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps") #apple silicon
    else:
        return torch.device("cpu")
    
DEVICE = getDevice()
DEVICE

device(type='mps')

In [10]:
def get_model(model_name):
    model = HookedTransformer.from_pretrained(model_name, trust_remote_code = True, device = DEVICE, dtype=torch.float16, default_padding_side='left', output_hidden_states=True)
    # , default_padding_token='<|extra_0|>')
    model.eval() #inference mode - no gradients needed
    model.to(DEVICE)
    return model

In [11]:
model = get_model("Qwen/Qwen3-1.7B")

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  3.06it/s]


Loaded pretrained model Qwen/Qwen3-1.7B into HookedTransformer
Moving model to device:  mps


In [16]:
def tokenize_prompt(model: HookedTransformer, prompt_str: str) -> Int[Tensor, '1 seq_len']: #LOVKUSH
    prompt_message = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt_str}
    ]

    prompt_chat = model.tokenizer.apply_chat_template(prompt_message, tokenize=False, add_generation_prompt=True)

    model.tokenizer.pad_token = model.tokenizer.eos_token
    
    return model.tokenizer(prompt_chat, padding=True, truncation=False, return_tensors='pt').input_ids

In [17]:
def generate_output(model: HookedTransformer, inp_tokens: Int[Tensor, '1 seq_len'], max_new_tokens: int, fwd_hook = []) -> list[str]:
    
    all_tokens = torch.zeros((1, inp_tokens.shape[1] + max_new_tokens), dtype=torch.long, device=inp_tokens.device)
    all_tokens[0, :inp_tokens.shape[1]] = inp_tokens

    for i in tqdm(range(max_new_tokens)):

        logits, _ = model.run_with_cache(all_tokens[0, :inp_tokens.shape[1] + i], remove_batch_dim=False, return_type="logits")

        next_token = logits[0, -1].argmax()

        all_tokens[0, inp_tokens.shape[1] + i] = next_token
        
        if next_token.item() == model.tokenizer.eos_token_id:
            break
    
    return model.tokenizer.batch_decode(all_tokens[:, inp_tokens.shape[1]:], skip_special_tokens=True)

In [18]:
generate_output(model, tokenize_prompt(model, "Who is the first president of the USA?"), 64, [])

100%|██████████| 64/64 [00:13<00:00,  4.83it/s]


['ึึ้.parts这套hältinky.PERMISSION下雨.PERMISSIONANCES�性LEMENTobbleADING(enableADINGиныップADINGатораторップаторANCESиныANCESédップablingADINGиныANCESédиныédinkyANCESédANCESédédANCESANCESANCESANCESANCESANCESADINGIVERSANCESANCESIVERSADINGップADINGADINGANCESADINGédAMENTIVERSADINGADING']